# Example XML Validation

## Simple. Powerful. Secure by default.

Python’s built-in XML parsers are **vulnerable by default**.  
One malicious file can cause memory exhaustion, stack overflows, or XXE attacks.

**FileAudit** solves this with a single, elegant function:

```python
validate_xml("file.xml")          # that's it
```

This notebook shows how **simple** it is to use and how **powerful** the protection is.

## 1. Installation

```bash
pip install fileaudit
```

> For this standalone demo we embed the complete source code below  
> so the notebook runs without any external package.

In [1]:
from fileaudit.xml_check import validate_xml, FileValidationError

## 2. Create sample XML files

To demonstrate the power of this secure programming API, we create one **safe** file and several **malicious** files that would crash or exhaust a normal Python XML parser.

In [2]:
from pathlib import Path

demo_dir = Path("xml_demo_files")
demo_dir.mkdir(exist_ok=True)

# ----------------------------------------------------------
# 1. Safe, perfectly valid XML
# ----------------------------------------------------------
safe_xml = '''<?xml version="1.0" encoding="UTF-8"?>
<catalog>
  <book id="bk101">
    <author>Gambardella, Matthew</author>
    <title>XML Developer's Guide</title>
    <genre>Computer</genre>
    <price>44.95</price>
  </book>
  <book id="bk102">
    <author>Ralls, Kim</author>
    <title>Midnight Rain</title>
    <genre>Fantasy</genre>
    <price>5.95</price>
  </book>
</catalog>'''

(demo_dir / "safe.xml").write_text(safe_xml, encoding="utf-8")

# ----------------------------------------------------------
# 2. Classic Billion Laughs / XXE style attack (DOCTYPE)
# ----------------------------------------------------------
billion_laughs = '''<?xml version="1.0"?>
<!DOCTYPE lolz [
  <!ENTITY lol "lol">
  <!ENTITY lol1 "&lol;&lol;&lol;&lol;&lol;&lol;&lol;&lol;&lol;&lol;">
  <!ENTITY lol2 "&lol1;&lol1;&lol1;&lol1;&lol1;&lol1;&lol1;&lol1;&lol1;&lol1;">
  <!ENTITY lol3 "&lol2;&lol2;&lol2;&lol2;&lol2;&lol2;&lol2;&lol2;&lol2;&lol2;">
]>
<lolz>&lol3;</lolz>'''

(demo_dir / "billion_laughs.xml").write_text(billion_laughs, encoding="utf-8")

# ----------------------------------------------------------
# 3. Extremely deep nesting (stack overflow)
# ----------------------------------------------------------
depth = 80
deep_xml = "<root>" + ("<a>" * depth) + "deep" + ("</a>" * depth) + "</root>"
(demo_dir / "deep_nesting.xml").write_text(deep_xml, encoding="utf-8")

# ----------------------------------------------------------
# 4. Attribute bomb
# ----------------------------------------------------------
attrs = " ".join(f'attr{i}="value{i}"' for i in range(1500))
attr_bomb = f'<root {attrs}/>'
(demo_dir / "attribute_bomb.xml").write_text(attr_bomb, encoding="utf-8")

# ----------------------------------------------------------
# 5. Huge text node
# ----------------------------------------------------------
huge_text = "<root>" + ("A" * 200_000) + "</root>"
(demo_dir / "huge_text.xml").write_text(huge_text, encoding="utf-8")

print("Created demo files:")
for f in sorted(demo_dir.iterdir()):
    print(f"  • {f.name:25} ({f.stat().st_size:,} bytes)")

Created demo files:
  • attribute_bomb.xml        (29,287 bytes)
  • billion_laughs.xml        (313 bytes)
  • deep_nesting.xml          (577 bytes)
  • huge_text.xml             (200,013 bytes)
  • safe.xml                  (371 bytes)


## 3. The simplest possible usage – Direct call

Just pass a file path.  
If the file is safe → returns `True`.  
If the file is dangerous → raises a clear `FileValidationError`.

In [3]:
print("=== Safe file ===")
result = validate_xml("xml_demo_files/safe.xml")
print(f"Result: {result}")
print("✅ File is safe – you can now parse it with the standard library\n")

=== Safe file ===
Result: True
✅ File is safe – you can now parse it with the standard library



In [4]:
print("=== Billion Laughs / DOCTYPE attack ===")
try:
    validate_xml("xml_demo_files/billion_laughs.xml")
except FileValidationError as e:
    print(f"🛡️  Blocked: {e}\n")

=== Billion Laughs / DOCTYPE attack ===
🛡️  Blocked: FileAudit Security Validation Failed - DOCTYPE declarations are not allowed.



In [5]:
print("=== Deep nesting attack ===")
try:
    validate_xml("xml_demo_files/deep_nesting.xml")
except FileValidationError as e:
    print(f"🛡️  Blocked: {e}\n")

=== Deep nesting attack ===
🛡️  Blocked: FileAudit Security Validation Failed - XML nesting depth exceeds 50



In [6]:
print("=== Attribute bomb ===")
try:
    validate_xml("xml_demo_files/attribute_bomb.xml")
except FileValidationError as e:
    print(f"🛡️  Blocked: {e}\n")

=== Attribute bomb ===
🛡️  Blocked: FileAudit Security Validation Failed - Too many attributes on element 'root'



In [7]:
print("=== Huge text node ===")
try:
    validate_xml("xml_demo_files/huge_text.xml")
except FileValidationError as e:
    print(f"🛡️  Blocked: {e}\n")

=== Huge text node ===
🛡️  Blocked: FileAudit Security Validation Failed - Text node exceeds limit



## 4. Decorator – protect a whole function with one line

The most elegant way to use FileAudit in real applications.

In [8]:
import xml.etree.ElementTree as ET #needed for below demo

In [9]:
@validate_xml
def process_xml(file_path):
    """This function is now automatically protected."""
    tree = ET.parse(file_path)
    return tree.getroot().tag

print("Calling process_xml with a safe file:")
print("→", process_xml("xml_demo_files/safe.xml"))

print("\nCalling process_xml with a malicious file:")
try:
    process_xml("xml_demo_files/billion_laughs.xml")
except FileValidationError as e:
    print(f"🛡️  Blocked before the function body even ran:\n   {e}")

Calling process_xml with a safe file:
→ catalog

Calling process_xml with a malicious file:
🛡️  Blocked before the function body even ran:
   FileAudit Security Validation Failed - DOCTYPE declarations are not allowed.


## 5. Custom limits – tune the protection

You can tighten (or relax) any limit.

In [10]:
# Very strict limits for untrusted uploads
print("Trying a normal file with very strict limits...")
try:
    validate_xml(
        "xml_demo_files/safe.xml",
        max_depth=3,          # only 3 levels deep allowed
        max_elements=5,       # max 5 elements
        max_attributes=2
    )
except FileValidationError as e:
    print(f"🛡️  Rejected even the safe file because limits are very strict:\n   {e}")

Trying a normal file with very strict limits...
🛡️  Rejected even the safe file because limits are very strict:
   FileAudit Security Validation Failed - XML contains more than 5 elements


## 6. Factory mode – reusable pre-configured validator

Perfect when many functions should share the same security policy.

In [11]:
# Create a strict validator once
strict_xml = validate_xml(
    max_depth=20,
    max_file_size=100_000,      # 100 KB
    max_elements=500,
    max_attributes=30,
    max_text_length=5_000
)

@strict_xml
def import_catalog(path):
    print(f"Importing {path} ...")
    return ET.parse(path)

print("Safe file with strict policy:")
import_catalog("xml_demo_files/safe.xml")
print("✅ Success\n")

print("Malicious file with the same strict policy:")
try:
    import_catalog("xml_demo_files/attribute_bomb.xml")
except FileValidationError as e:
    print(f"🛡️  Blocked: {e}")

Safe file with strict policy:
Importing xml_demo_files/safe.xml ...
✅ Success

Malicious file with the same strict policy:
🛡️  Blocked: FileAudit Security Validation Failed - Too many attributes on element 'root'


## Why this is powerful

| Feature | What it protects against | Effort for you |
|---------|--------------------------|----------------|
| **One-liner API** | – | `validate_xml(path)` |
| **DOCTYPE rejection** | XXE, Billion Laughs, entity expansion | Automatic |
| **Depth limit** | Stack overflow / deep nesting | Automatic |
| **Element & attribute limits** | Attribute bombs, DoS | Automatic |
| **Text size limit** | Memory exhaustion | Automatic |
| **HTTPS-only remote files** | SSRF & scheme downgrade | Automatic |
| **GZip bomb protection** | Decompression bombs | Automatic |
| **Decorator & factory** | Secure whole functions / policies | One line |

### You get enterprise-grade XML security with almost zero code.

## Notebook Clean up 

In [12]:
import shutil
shutil.rmtree("xml_demo_files", ignore_errors=True)
print("Demo files removed.")

Demo files removed.


---
(c) By Maikel Mardjan (nocomplexity.com) and all contributors.
Creative Commons License. This work is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License.

[**FileAudit**](https://pypi.org/project/fileaudit) – Make XML processing safe by default.  
